In [12]:
from pyspark.sql.functions import *

df_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

StatementMeta(divanshpool, 17, 13, Finished, Available, Finished, False)

In [21]:
from pyspark.sql.functions import rand

stream_df = df_stream.select(
    concat(lit("O"), col("value")),
    when(rand() > 0.5, "Laptop").otherwise("Mobile").alias("product"),
    (rand() * 50000).cast("int").alias("price"),
    (rand() * 3 + 1).cast("int").alias("quantity"),
    lit("Bangalore").alias("city"),
    current_timestamp().alias("event_time")
)

StatementMeta(divanshpool, 17, 22, Finished, Available, Finished, False)

In [13]:
# stream_df = df_stream.select(
#     concat(lit("O"), col("value")).alias("order_id"),
#     when(col("value") % 3 == 0, "Laptop")
#     .when(col("value") % 3 == 1, "Mobile")
#     .otherwise("Tablet").alias("product"),
#     (col("value") * 100).alias("price"),
#     (col("value") % 3 + 1).alias("quantity"),
#     ((col("value") * 100) * (col("value") % 3 + 1)).alias("total_amount"),
#     when(col("value") % 2 == 0, "Bangalore")
#     .otherwise("Delhi").alias("city"),
#     lit("UPI").alias("payment_method"),
#     current_timestamp().alias("event_time")
# )

StatementMeta(divanshpool, 17, 14, Finished, Available, Finished, False)

In [22]:
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://<contianer>@<storage_account>.dfs.core.windows.net/bronze/checkpoints/stream") \
    .start("abfss://<contianer>@<storage_account>.dfs.core.windows.net/bronze/streaming/")

StatementMeta(divanshpool, 17, 23, Finished, Available, Finished, False)

In [23]:
query.status

StatementMeta(divanshpool, 17, 24, Finished, Available, Finished, False)

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': True}

In [24]:
spark.read.format("delta") \
    .load("abfss://<contianer>@<storage_account>.dfs.core.windows.net/bronze/streaming/") \
    .show(10)

StatementMeta(divanshpool, 17, 25, Finished, Available, Finished, False)

+--------+-------+------+--------+------------+---------+--------------+--------------------+
|order_id|product| price|quantity|total_amount|     city|payment_method|          event_time|
+--------+-------+------+--------+------------+---------+--------------+--------------------+
|   O1134| Laptop|113400|       1|      113400|Bangalore|           UPI|2026-04-04 06:57:...|
|   O1150| Mobile|115000|       2|      230000|Bangalore|           UPI|2026-04-04 06:57:...|
|   O1166| Tablet|116600|       3|      349800|Bangalore|           UPI|2026-04-04 06:57:...|
|   O1182| Laptop|118200|       1|      118200|Bangalore|           UPI|2026-04-04 06:57:...|
|   O1198| Mobile|119800|       2|      239600|Bangalore|           UPI|2026-04-04 06:57:...|
|   O1411| Mobile|141100|       2|      282200|    Delhi|           UPI|2026-04-04 06:58:...|
|   O1427| Tablet|142700|       3|      428100|    Delhi|           UPI|2026-04-04 06:58:...|
|   O1443| Laptop|144300|       1|      144300|    Delhi|   

In [9]:
query.stop()

StatementMeta(divanshpool, 17, 10, Finished, Available, Finished, False)

In [25]:
df_stream_read = spark.read.format("delta") \
    .load("abfss://<contianer>@<storage_account>.dfs.core.windows.net/bronze/streaming/")

clean_stream = df_stream_read.dropna()

clean_stream.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://<contianer>@<storage_account>.dfs.core.windows.net/silver/stream_clean/")

StatementMeta(divanshpool, 17, 26, Finished, Available, Finished, False)

In [26]:
from pyspark.sql.functions import sum

gold_stream = clean_stream.groupBy("product", "city") \
    .agg(sum("total_amount").alias("total_sales"))

gold_stream.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://<contianer>@<storage_account>.dfs.core.windows.net/gold/stream_summary/")

StatementMeta(divanshpool, 17, 27, Finished, Available, Finished, False)

In [27]:
from pyspark.sql.functions import sum

batch_df = spark.read.format("delta") \
    .load("abfss://<contianer>@<storage_account>.dfs.core.windows.net/gold/sales_summary/")

stream_df = spark.read.format("delta") \
    .load("abfss://<contianer>@<storage_account>.dfs.core.windows.net/gold/stream_summary/")

final_df = batch_df.union(stream_df)

final_df.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://<contianer>@<storage_account>.dfs.core.windows.net/gold/final_sales/")

StatementMeta(divanshpool, 17, 28, Finished, Available, Finished, False)